# AI Research Assistant Agent — End-to-End Notebook

Implements a research assistant with LangChain `create_agent`, OpenRouter (`openrouter/free`), PyMuPDF PDF reading, Tavily web search, a Python calculator, execution traces, session memory, and a classical rule-based baseline.

## Setup
1. In Google Colab, open **Secrets** and add `OPENROUTER_API_KEY` and `TAVILY_API_KEY`.
2. Enable notebook access to those secrets.
3. Run cells from top to bottom and upload a research-paper PDF when prompted.
4. Do not hard-code API keys. `openrouter/free` is a routing identifier, not one fixed model.
5. OpenRouter HTTP 429 errors indicate a quota/rate-limit problem; no successful output should be fabricated.

In [ ]:
%pip -q install -U langchain langchain-core langchain-openai pymupdf tavily-python

In [ ]:
import os, json, time, statistics, re
from pathlib import Path
from collections import defaultdict
from typing import List
import fitz
from IPython.display import display, Markdown
from google.colab import files, userdata

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
TAVILY_API_KEY = userdata.get("TAVILY_API_KEY")
if not OPENROUTER_API_KEY:
    raise ValueError("Add OPENROUTER_API_KEY to Google Colab Secrets.")
if not TAVILY_API_KEY:
    raise ValueError("Add TAVILY_API_KEY to Google Colab Secrets.")
print("API secrets loaded; values are not displayed.")

In [ ]:
from langchain_openai import ChatOpenAI

MODEL_ID = "openrouter/free"
llm = ChatOpenAI(
    model=MODEL_ID,
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    temperature=0,
    timeout=90,
    max_retries=1,
)
print("Configured OpenRouter routing identifier:", MODEL_ID)

## Upload the research paper and extract its text

In [ ]:
uploaded = files.upload()
pdf_files = [name for name in uploaded if name.lower().endswith(".pdf")]
if not pdf_files:
    raise ValueError("Upload a PDF file and rerun this cell.")
PDF_PATH = pdf_files[0]
print("Selected PDF:", PDF_PATH)

def extract_pdf_text(pdf_path, max_chars=50000):
    if not Path(pdf_path).exists():
        raise FileNotFoundError(pdf_path)
    page_texts = []
    with fitz.open(pdf_path) as doc:
        for page_no, page in enumerate(doc, start=1):
            text = page.get_text("text")
            if text.strip():
                page_texts.append(f"--- Page {page_no} ---\n{text}")
    result = "\n\n".join(page_texts).strip()
    return result[:max_chars] if result else (
        "No selectable text extracted. The PDF may be scanned and require OCR."
    )

PDF_TEXT = extract_pdf_text(PDF_PATH)
print(f"Extracted {len(PDF_TEXT):,} characters.")
print(PDF_TEXT[:4000])

## Define the PDF reader, Tavily web search, and calculator tools

In [ ]:
from langchain_core.tools import tool
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def read_uploaded_research_paper() -> str:
    """Return extracted text from the uploaded research paper."""
    return PDF_TEXT

@tool
def search_research_information(query: str) -> str:
    """Search the web and return result titles, URLs, and text snippets."""
    if not query.strip():
        return "Search query was empty."
    data = tavily_client.search(
        query=query.strip(), max_results=5, search_depth="basic",
        include_answer=False
    )
    results = data.get("results", [])
    if not results:
        return "No relevant search results were returned."
    return "\n\n".join(
        f"Result {i}\nTitle: {r.get('title', 'N/A')}\nURL: {r.get('url', 'N/A')}\nContent: {r.get('content', 'N/A')}"
        for i, r in enumerate(results, 1)
    )

@tool
def research_calculator(baseline_value: float, comparison_value: float) -> dict:
    """Calculate absolute difference and signed relative percentage change from baseline to comparison."""
    baseline_value = float(baseline_value)
    comparison_value = float(comparison_value)
    diff = comparison_value - baseline_value
    pct = None if baseline_value == 0 else diff / abs(baseline_value) * 100
    return {
        "baseline_value": baseline_value,
        "comparison_value": comparison_value,
        "absolute_difference": abs(diff),
        "signed_difference": diff,
        "relative_change_percent": pct,
        "note": "A numerical difference does not prove that two study metrics are comparable."
    }

tools = [read_uploaded_research_paper, search_research_information, research_calculator]
for t in tools:
    print(t.name, "-", t.description)

## Create the LangChain agent

In [ ]:
from langchain.agents import create_agent

SYSTEM_PROMPT = """
You are an AI Research Assistant.
- Read the uploaded paper when the request concerns it.
- Use web search when external research is requested or needed.
- Use the calculator when valid numerical inputs are available and useful.
- Never invent titles, authors, citations, URLs, datasets, or performance metrics.
- Distinguish uploaded-document information from external search results.
- If a metric is not found in the extracted paper text, say so.
- Do not claim metrics from different studies are comparable unless their definitions,
  datasets, and evaluation protocols justify comparison.
- State uncertainty clearly and provide source URLs returned by web search when relevant.
- Produce a concise, structured answer.
"""

agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)
print("Agent created.")

## Execution trace helpers

In [ ]:
def message_content(message):
    content = getattr(message, "content", "")
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "\n".join(
            item.get("text", str(item)) if isinstance(item, dict) else str(item)
            for item in content
        )
    return str(content)

def display_execution_trace(response, test_name="Agent Run",
                            max_tool_chars=1500, max_ai_chars=3000):
    messages = response.get("messages", [])
    tool_names = []
    print("=" * 80)
    print("AI RESEARCH ASSISTANT — EXECUTION TRACE")
    print("Test:", test_name, "| Messages:", len(messages))
    print("=" * 80)
    for i, msg in enumerate(messages, 1):
        msg_type = getattr(msg, "type", msg.__class__.__name__)
        print(f"\n[{i}] Message type: {msg_type}")
        calls = getattr(msg, "tool_calls", None) or []
        for call in calls:
            name = call.get("name", "unknown")
            tool_names.append(name)
            print("TOOL CALL:", name)
            print("ARGUMENTS:", json.dumps(call.get("args", {}), ensure_ascii=False))
        content = message_content(msg)
        if content:
            limit = max_ai_chars if msg_type == "ai" else max_tool_chars
            print("CONTENT:\n", content[:limit] + ("\n...[truncated]" if len(content) > limit else ""))
    print("\nTotal tool calls:", len(tool_names))
    print("Tool names:", tool_names)

def extract_final_ai_response(response):
    for msg in reversed(response.get("messages", [])):
        if getattr(msg, "type", "") == "ai":
            text = message_content(msg).strip()
            if text:
                return text
    return "No final AI response found."

def invoke_research_agent(query, show_trace=True):
    result = agent.invoke({"messages": [{"role": "user", "content": query}]})
    if show_trace:
        display_execution_trace(result, query)
    return result

## Run the main research task and create the actual final response

In [ ]:
research_query = (
    "Analyse the uploaded research paper. Identify its title and explain its main "
    "lane-detection methodology. State whether the extracted text reports an explicit "
    "quantitative performance metric. If useful, search for related research, but do not "
    "invent or treat unverified external metrics as facts."
)

research_response = None
research_final_text = None
try:
    research_response = invoke_research_agent(research_query, show_trace=True)
    research_final_text = extract_final_ai_response(research_response)
    print("\nFINAL RESEARCH RESPONSE")
    display(Markdown(research_final_text))
except Exception as exc:
    print("Agent execution failed:", type(exc).__name__, str(exc))
    print("If this is HTTP 429, check OpenRouter quota/rate limits and retry when available.")

## Screenshot 5 output — use the real response, not a fabricated example

In [ ]:
if research_response is None:
    print("No successful research_response is available. Run the main research task successfully first.")
else:
    research_final_text = extract_final_ai_response(research_response)
    print("=" * 80)
    print("AI RESEARCH ASSISTANT — FINAL RESEARCH OUTPUT")
    print("=" * 80)
    display(Markdown(research_final_text))
    print("=" * 80)
    print("STATUS: Actual agent response displayed.")

## Test the calculator separately

In [ ]:
calculator_test = research_calculator.invoke({
    "baseline_value": 0.82,
    "comparison_value": 0.87
})
print(json.dumps(calculator_test, indent=2))
print("These are illustrative test values, not metrics from the uploaded paper.")

## Session-based conversational memory (temporary in-memory storage)

In [ ]:
conversation_store = defaultdict(list)

def invoke_memory_agent(session_id: str, user_query: str, show_trace=False):
    if not session_id:
        raise ValueError("session_id must be a non-empty string.")
    prior = conversation_store[session_id]
    result = agent.invoke({
        "messages": list(prior) + [{"role": "user", "content": user_query}]
    })
    conversation_store[session_id] = result.get("messages", [])
    if show_trace:
        display_execution_trace(result, f"Memory session: {session_id}")
    return result

# Example (these make live LLM calls; run when API quota is available):
# first = invoke_memory_agent("student_a", "Summarise the uploaded paper.")
# follow_up = invoke_memory_agent("student_a", "What are its main limitations?")
# Memory is lost when the Colab runtime resets.

## Classical keyword-based baseline

In [ ]:
def select_classical_tool(query: str) -> str:
    q = query.lower()
    calculator_terms = [
        "calculate", "percentage change", "relative change",
        "difference between", "compare values", "how much higher", "how much lower"
    ]
    search_terms = [
        "search", "find a study", "related study", "web", "external source",
        "recent research", "look up", "online"
    ]
    document_terms = [
        "uploaded paper", "uploaded pdf", "this paper", "the paper",
        "methodology", "abstract", "limitations", "document", "pdf"
    ]
    if any(x in q for x in calculator_terms):
        return "research_calculator"
    if any(x in q for x in search_terms):
        return "search_research_information"
    if any(x in q for x in document_terms):
        return "read_uploaded_research_paper"
    return "read_uploaded_research_paper"

def execute_classical_tool(tool_name: str, query: str) -> str:
    if tool_name == "read_uploaded_research_paper":
        return read_uploaded_research_paper.invoke({})
    if tool_name == "search_research_information":
        return search_research_information.invoke({"query": query})
    if tool_name == "research_calculator":
        numbers = re.findall(r"[-+]?(?:\d*\.\d+|\d+)", query)
        if len(numbers) < 2:
            return "Calculator selected, but two numerical inputs could not be extracted."
        return json.dumps(research_calculator.invoke({
            "baseline_value": float(numbers[0]),
            "comparison_value": float(numbers[1])
        }), indent=2)
    return "No tool selected."

def classical_research_agent(query: str) -> dict:
    start = time.perf_counter()
    selected = select_classical_tool(query)
    try:
        output = execute_classical_tool(selected, query)
        success = bool(output) and not output.startswith("Calculator selected, but")
    except Exception as exc:
        output, success = f"{type(exc).__name__}: {exc}", False
    result = {
        "query": query,
        "selected_tool": selected,
        "success": success,
        "latency_seconds": time.perf_counter() - start,
        "output": output,
    }
    print(json.dumps(result, indent=2, ensure_ascii=False)[:10000])
    return result

# Example:
# baseline_result = classical_research_agent("Explain the methodology used in the uploaded paper.")

## Optional ten-query evaluation — run only when API quota is available

In [ ]:
evaluation_queries = [
    "Explain the methodology used in the uploaded paper.",
    "Summarise the abstract of the uploaded paper.",
    "What are the main limitations of the uploaded paper?",
    "Search for a related study on lane detection.",
    "Find external research on deep-learning lane detection.",
    "Search for evaluation metrics used in lane-detection research.",
    "Calculate the difference between 0.82 and 0.87.",
    "What does the uploaded paper say about lane detection?",
    "Find a related study and summarise its methodology.",
    "Explain the uploaded paper and identify information that is missing.",
]

def run_evaluation(queries: List[str]):
    rows = []
    for i, query in enumerate(queries, 1):
        print(f"\n===== Query {i}/{len(queries)} =====")
        expected = select_classical_tool(query)
        classical = classical_research_agent(query)

        start = time.perf_counter()
        modern_names, modern_success, error = [], False, None
        try:
            modern = agent.invoke({"messages": [{"role": "user", "content": query}]})
            for msg in modern.get("messages", []):
                for call in (getattr(msg, "tool_calls", None) or []):
                    modern_names.append(call.get("name", "unknown"))
            modern_success = bool(extract_final_ai_response(modern).strip())
        except Exception as exc:
            error = f"{type(exc).__name__}: {exc}"
            print("Modern-agent error:", error)
        modern_latency = time.perf_counter() - start

        rows.append({
            "query": query,
            "expected_tool": expected,
            "classical_tool": classical["selected_tool"],
            "classical_action_correct": classical["selected_tool"] == expected,
            "classical_success": classical["success"],
            "classical_latency_seconds": classical["latency_seconds"],
            "modern_tool_calls": ", ".join(modern_names),
            "modern_action_correct": expected in modern_names,
            "modern_success": modern_success,
            "modern_latency_seconds": modern_latency,
            "modern_error": error,
        })
    return rows

# To run: evaluation_results = run_evaluation(evaluation_queries)
# This makes multiple API calls. Failed calls must remain recorded; do not invent results.

In [ ]:
def summarise_evaluation(rows):
    if not rows:
        raise ValueError("No evaluation rows supplied.")
    n = len(rows)
    classical_times = [r["classical_latency_seconds"] for r in rows]
    modern_times = [r["modern_latency_seconds"] for r in rows]
    summary = {
        "number_of_queries": n,
        "classical_action_selection_accuracy_percent": 100 * sum(r["classical_action_correct"] for r in rows) / n,
        "classical_execution_success_percent": 100 * sum(r["classical_success"] for r in rows) / n,
        "classical_mean_latency_seconds": statistics.mean(classical_times),
        "classical_median_latency_seconds": statistics.median(classical_times),
        "modern_action_selection_accuracy_percent": 100 * sum(r["modern_action_correct"] for r in rows) / n,
        "modern_execution_success_percent": 100 * sum(r["modern_success"] for r in rows) / n,
        "modern_mean_latency_seconds": statistics.mean(modern_times),
        "modern_median_latency_seconds": statistics.median(modern_times),
        "caution": "Action-selection correctness does not measure factual accuracy. Inspect errors and traces."
    }
    print(json.dumps(summary, indent=2))
    return summary

# After evaluation:
# import pandas as pd
# display(pd.DataFrame(evaluation_results))
# evaluation_summary = summarise_evaluation(evaluation_results)

## Final checklist before sharing

- Run the notebook from top to bottom in a fresh Colab runtime.
- Store keys in Colab Secrets; never commit API keys.
- Test PDF extraction and calculator independently.
- Run the live agent task only when OpenRouter quota is available.
- For Screenshot 5, run the main research task successfully and capture the actual final response.
- Do not present unverified external metrics as facts.
- If you run evaluation, report results from that exact run and document HTTP 429 failures.
- Share the notebook with the instructor using a link with appropriate access.